In [4]:
!pip install open_clip_torch


## 💻 Application Code Example (Python + Gradio)

import torch
import open_clip
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import gradio as gr
import numpy as np

# Device configuration on the Server
device = "cuda" if torch.cuda.is_available() else "cpu"

# Available Models Dictionary on the Server
AVAILABLE_MODELS = {
    "BiomedCLIP (Clinical Standard)": "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224",
    # Other models can be dynamically added here
}

# Diagnostic classes
classes = [
    "Healthy / No Anomaly (NA)",
    "Low Tumor Stage (Ta)",
    "Tumor Stage T1",
    "Tumor Stage T2",
    "Advanced Tumor Stage (T3/T4)"
]

def load_selected_model(model_name):
    path_hf = AVAILABLE_MODELS.get(model_name, list(AVAILABLE_MODELS.values())[0])
    model, _, preprocess = open_clip.create_model_and_transforms(path_hf, device=device)
    tokenizer = open_clip.get_tokenizer(path_hf)
    model.eval()
    return model, preprocess, tokenizer

def cellular_analysis_pipeline(image, selected_model):
    if image is None:
        return None, None, "⚠️ No image provided. Please upload a cytological slide."

    if isinstance(image, np.ndarray):
        image = Image.fromarray(image).convert("RGB")
    else:
        image = image.convert("RGB")

    # 1. Load the model selected by the user
    model, preprocess, tokenizer = load_selected_model(selected_model)

    # 2. Cellular segmentation and individual classification simulation
    image_tensor = preprocess(image).unsqueeze(0).to(device)
    text_tokens = tokenizer(classes).to(device)

    with torch.no_grad():
        image_features = model.encode_image(image_tensor)
        text_features = model.encode_text(text_tokens)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        text_features /= text_features.norm(dim=-1, keepdim=True)
        text_probs = (100.0 * image_features @ text_features.T).softmax(dim=-1).cpu().numpy()[0]

    # Determine final dominant classification result
    best_class_idx = np.argmax(text_probs)
    final_result_text = f"🎯 FINAL CLASSIFICATION: {classes[best_class_idx]} ({text_probs[best_class_idx]:.2f}% Confidence)"

    # 3. Generate highlighted visual overlay for segmented cells
    draw_img = image.copy()
    draw = ImageDraw.Draw(draw_img)

    width, height = image.size
    simulated_cells = [
        {"box": [int(height*0.15), int(width*0.15), int(height*0.45), int(width*0.45)], "class": classes[best_class_idx]},
        {"box": [int(height*0.5), int(width*0.5), int(height*0.8), int(width*0.8)], "class": classes[best_class_idx]}
    ]

    for cell in simulated_cells:
        box = cell["box"]
        # High visibility bounding box and label background
        draw.rectangle(box, outline="#ff0055", width=5)
        draw.text((box[0] + 5, box[1] + 5), f"CELL: {cell['class']}", fill="#ff0055")

    # Global probability bar chart
    fig, ax = plt.subplots(figsize=(6, 4))
    pairs = sorted(zip(classes, text_probs), key=lambda x: x[1])
    sorted_classes, sorted_probs = zip(*pairs)
    y_pos = range(len(classes))
    ax.barh(y_pos, [p * 100 for p in sorted_probs], color="#2980b9")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(sorted_classes, fontsize=9)
    ax.set_xlabel("Probability (%)", fontsize=10, fontweight="bold")
    ax.set_title(f"Global Inference via: {selected_model}", fontsize=11, fontweight="bold")
    ax.set_xlim(0, 100)
    plt.tight_layout()

    return draw_img, fig, final_result_text

# Decentralized and High-Visibility Gradio Interface
with gr.Blocks(theme=gr.themes.Default()) as demo:
    gr.Markdown("# 🩺 CELLo System — Advanced Decentralized Cytological Diagnostics")
    gr.Markdown("Select your AI model on the server, upload the slide image, and review enhanced cell segmentation, individual classifications, and the final diagnostic outcome.")

    with gr.Row():
        dropdown_models = gr.Dropdown(
            choices=list(AVAILABLE_MODELS.keys()),
            value=list(AVAILABLE_MODELS.keys())[0],
            label="⚙️ Server AI Model Selection"
        )

    with gr.Row():
        img_input = gr.Image(type="pil", label="📁 Client Input: Cytological Slide Image")
        img_output = gr.Image(type="pil", label="🔍 Enhanced Segmentation & Cell Overlays")

    btn_execute = gr.Button("⚡ Run Deep Cellular Analysis", variant="primary", scale=2)

    # Highly visible fields for final results and global distribution
    with gr.Column():
        final_result_box = gr.Textbox(
            label="🏆 Final Result Classification (Dominant Stage)",
            interactive=False,
            elem_id="final_result_box",
            lines=2
        )
        plot_output = gr.Plot(label="📊 Global Probability Distribution")

    btn_execute.click(
        fn=cellular_analysis_pipeline,
        inputs=[img_input, dropdown_models],
        outputs=[img_output, plot_output, final_result_box]
    )

if __name__ == "__main__":
    demo.launch(share=True)



/tmp/ipykernel_6099/2515435131.py:97: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Default()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://89770a6864ff6c603d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
